# Unified Model GRPO Reinforcement Learning Training

This notebook implements **GRPO (Group Relative Policy Optimization)** reinforcement learning to align the **Unified 1-Model Logic Reasoning Pipeline** using deterministic feedback from the **Z3 Symbolic Solver**.

It implements custom reward functions to penalize common failure modes: XML parsing errors, logical proof errors (Z3 solver disagreement), fact hallucination, empty MCQ option constraints, and lười biếng using propositional logic instead of First-Order Logic (FOL) for quantifier-heavy premises.

In [1]:
print("Installing other dependencies...")
!pip install -q -U trl transformers accelerate peft z3-solver rapidfuzz
print('Dependencies ready! Please remember to restart the Jupyter kernel if needed.')

Installing other dependencies...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/HipMarker-1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/rpdTracer-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Dependencies ready! Please remember to restart the Jupyter kernel if needed.


In [ ]:

# ========================================================
# STEP 1: Import Dependencies & Verify Setup
# ========================================================
import os
import re
import sys
import json
import gc
from pathlib import Path
import argparse
import torch

if not hasattr(torch, 'float8_e8m0fnu'):
    setattr(torch, 'float8_e8m0fnu', torch.float32)

# Workaround for vllm StructuredOutputsParams import error in newer TRL / older vLLM environments
import sys
import importlib
try:
    trl_import_utils = importlib.import_module('trl.import_utils')
    trl_import_utils.is_vllm_available = lambda: False
    trl_import_utils._vllm_available = False
except Exception:
    pass
sys.modules['vllm'] = None
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
# --- Patch FSDPModule for PyTorch 2.5.1 compatibility ---
import sys
import torch
try:
    import torch.distributed.fsdp as fsdp
    if not hasattr(fsdp, 'FSDPModule'):
        from torch.distributed._composable.fsdp import FSDPModule
        fsdp.FSDPModule = FSDPModule
        sys.modules['torch.distributed.fsdp'].FSDPModule = FSDPModule
except ImportError:
    pass
# -----------------------------------------------------

from trl import GRPOConfig, GRPOTrainer

from transformers import TrainerCallback

class GRPOLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f'\n=== Step {state.global_step} ===')
            if 'loss' in logs:
                print(f'  - Loss: {logs["loss"]:.4f}')
            if 'learning_rate' in logs:
                print(f'  - LR: {logs["learning_rate"]:.2e}')
            if 'objective/kl' in logs:
                print(f'  - KL Divergence: {logs["objective/kl"]:.4f}')
            if 'completions/lengths' in logs:
                print(f'  - Avg Completion Length: {logs["completions/lengths"]:.1f} tokens')
            
            reward_keys = [k for k in logs.keys() if k.startswith("rewards/")]
            if reward_keys:
                print('  - Rewards Breakdown:')
                for k in sorted(reward_keys):
                    print(f'    * {k.replace("rewards/", "")}: {logs[k]:.4f}')
            print('==============================\n')

from huggingface_hub import login
import contextlib
import io

# Try importing Z3
try:
    from z3.z3 import *
except ImportError:
    from z3 import *

# Try importing rapidfuzz safely
try:
    from rapidfuzz import fuzz
except ImportError:
    fuzz = None

# Login Hugging Face
HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN)

# Safe repo cloning and directory changing
repo_url = "https://github.com/NguyenAn05/exact-2026.git"
repo_dir = "exact-2026"

if Path(os.getcwd()).name == repo_dir:
    print(f"Already inside the repository directory: {os.getcwd()}")
else:
    if not os.path.exists(repo_dir):
        print(f"Cloning repository from {repo_url}...")
        os.system(f"git clone {repo_url}")
    os.chdir(repo_dir)
    print(f"Changed working directory to: {os.getcwd()}")

print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')

Cloning repository from https://github.com/NguyenAn05/exact-2026.git...


Cloning into 'exact-2026'...


Changed working directory to: /workspace/exact-2026
CUDA Available: True
GPU Device: AMD Instinct MI300X VF


In [3]:
# ========================================================
# STEP 2: Configure GRPO Arguments
# ========================================================
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_id', type=str, default='Qwen/Qwen2.5-7B-Instruct')
    parser.add_argument('--sft_adapter_dir', type=str, default='NguyenAn05/qwen2.5-type1-sft-lora')
    parser.add_argument('--dataset_id', type=str, default='NguyenAn05/exact_2026_model1_fol_translator')
    parser.add_argument('--output_dir', type=str, default='models/qwen2.5-type1-grpo-lora')
    parser.add_argument('--hub_model_id', type=str, default='NguyenAn05/qwen2.5-type1-grpo-lora')
    parser.add_argument('--seed', type=int, default=42)
    return parser.parse_args()

sys.argv = ['']
args = parse_args()
print('GRPO Training configurations initialized!')
print(f'SFT Warm-Start Base Adapter: {args.sft_adapter_dir}')
print(f'Hub Model ID: {args.hub_model_id}')

GRPO Training configurations initialized!
SFT Warm-Start Base Adapter: NguyenAn05/qwen2.5-type1-sft-lora
Hub Model ID: NguyenAn05/qwen2.5-type1-grpo-lora


In [4]:
# ========================================================
# STEP 3: XML Parsing and Z3 Sandbox Executor
# ========================================================
def parse_xml_response(text: str) -> dict:
    """Parses the XML format generated by the Model (Singular version)."""
    premises_fol = []
    fol_match = re.search(r"<premises_fol>(.*?)</premises_fol>", text, re.DOTALL)
    if fol_match:
        fol_content = fol_match.group(1).strip()
        for line in fol_content.split("\n"):
            line = line.strip()
            if not line:
                continue
            line = re.sub(r"^\d+\.\s*", "", line)
            premises_fol.append(line)
            
    questions = []
    q_match = re.search(r"<question_output>(.*?)</question_output>", text, re.DOTALL)
    if q_match:
        q_content = q_match.group(1).strip()
        
        used_match = re.search(r"<premises_used>(.*?)</premises_used>", q_content, re.DOTALL)
        premises_used = []
        if used_match:
            try:
                premises_used = json.loads(used_match.group(1).strip())
            except Exception:
                premises_used = [int(n) for n in re.findall(r"\d+", used_match.group(1))]
                
        z3_match = re.search(r"<z3_code>(.*?)</z3_code>", q_content, re.DOTALL)
        z3_code = z3_match.group(1).strip() if z3_match else ""
        
        ans_match = re.search(r"<answer>(.*?)</answer>", q_content, re.DOTALL)
        answer = ans_match.group(1).strip() if ans_match else ""
        
        exp_match = re.search(r"<explanation>(.*?)</explanation>", q_content, re.DOTALL)
        explanation = exp_match.group(1).strip() if exp_match else ""
        
        questions.append({
            "id": 1,
            "premises_used": premises_used,
            "z3_code": z3_code,
            "answer": answer,
            "explanation": explanation
        })
        
    return {
        "premises_fol": premises_fol,
        "questions": questions
    }

def check_property(solver, prop):
    solver.set("timeout", 500)
    solver.push()
    solver.add(Not(prop))
    res = solver.check()
    solver.pop()
    if res == unsat:
        return "Yes"
        
    solver.push()
    solver.add(prop)
    res = solver.check()
    solver.pop()
    if res == unsat:
        return "No"
        
    return "Uncertain"

def check_mcq(solver, options_dict):
    solver.set("timeout", 500)
    proved = []
    for opt, opt_expr in options_dict.items():
        solver.push()
        solver.add(Not(opt_expr))
        res = solver.check()
        solver.pop()
        if res == unsat:
            proved.append(opt)
            
    if len(proved) >= 1:
        return proved[0]
    return "Uncertain"

def run_z3_code(z3_code_str: str) -> str:
    globals_dict = {
        'check_mcq': check_mcq,
        'check_property': check_property,
        '__builtins__': __builtins__
    }
    
    try:
        import z3 as z3_mod
    except ImportError as e:
        return f"Error: Z3 module not found ({e})"
            
    for name in dir(z3_mod):
        if not name.startswith('__'):
            globals_dict[name] = getattr(z3_mod, name)
            
    stdout_capture = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(z3_code_str, globals_dict)
    except Exception as e:
        return f"Error: {e}"
        
    output = stdout_capture.getvalue()
    match = re.search(r"Answer:\s*(\w+)", output, re.IGNORECASE)
    if match:
        return match.group(1)
    match_comma = re.search(r"Answer\s*,\s*(\w+)", output, re.IGNORECASE)
    if match_comma:
        return match_comma.group(1)
    return "Uncertain"

In [ ]:
# ========================================================
# STEP 4: Define GRPO Reward and Penalty Functions
# ========================================================
import concurrent.futures
import z3
try:
    import z3.z3 as z3_mod
except ImportError:
    import z3 as z3_mod

def xml_format_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    1. Format Reward: +1.0 if XML parses and has questions, -1.5 if parsing fails
    """
    rewards = []
    for completion in completions:
        parsed = parse_xml_response(completion)
        if parsed and parsed.get("questions"):
            rewards.append(1.0)
        else:
            rewards.append(-1.5) # Strong penalty for formatting errors
    return rewards

# Thread-safe Z3 execution (collects prints locally per thread instead of redirecting system stdout)
def run_z3_code_thread_safe(z3_code_str: str) -> str:
    output_lines = []
    # Custom print function that collects prints locally
    def custom_print(*args, **kwargs):
        output_lines.append(" ".join(str(x) for x in args))
    globals_dict = {
        'check_mcq': check_mcq,
        'check_property': check_property,
        'print': custom_print,
        '__builtins__': __builtins__,
        'z3': z3_mod
    }

    # Inject all symbols from z3 module for execution
    for name in dir(z3_mod):
        if not name.startswith('__'):
            globals_dict[name] = getattr(z3_mod, name)

    try:
        exec(z3_code_str, globals_dict)
    except Exception as e:
        return f"Error: {e}"

    output = "\n".join(output_lines)

    import re
    match = re.search(r"Answer:\s*(\w+)", output, re.IGNORECASE)

    if match:
        return match.group(1)

    match_comma = re.search(r"Answer\s*,\s*(\w+)", output, re.IGNORECASE)
    if match_comma:
        return match_comma.group(1)

    return "Uncertain"

def z3_accuracy_reward_func(prompts, completions, messages, **kwargs) -> list[float]:
    """
    2. Z3 Execution & Logical Accuracy Reward (Parallelized):

       - Correct proved answer: +4.0

       - Incorrect proved answer: -3.0 (Severe penalty)

       - Compilation crash / Timeout (0.5s): -2.0

    Includes: Option 1 mapping (uncertain -> no for Yes/No questions to match buggy GT labels).
    """
    rewards = []
    for completion, msg_list in zip(completions, messages):
        assistant_text = next((m['content'] for m in msg_list if m['role'] == 'assistant'), '')
        gt_parsed = parse_xml_response(assistant_text)
        gt_questions = gt_parsed.get('questions', [])
        gt_q_map = {q['id']: q['answer'] for q in gt_questions}

        parsed = parse_xml_response(completion)
        llm_questions = parsed.get('questions', [])

        if not llm_questions:
            rewards.append(-3.0) # No questions parsed
            continue

        # Collect Z3 codes to run
        z3_tasks = []
        for q in llm_questions:
            z3_tasks.append(q.get('z3_code', ''))

        # Run all Z3 codes concurrently using a ThreadPoolExecutor (max 16 workers)
        with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
            z3_results = list(executor.map(run_z3_code_thread_safe, z3_tasks))

        total_score = 0.0
        for i, q in enumerate(llm_questions):
            q_id = q['id']
            gt_ans = gt_q_map.get(q_id, '').strip().lower()
            z3_result = z3_results[i]

            if not z3_tasks[i]:
                total_score -= 2.0
                continue

            is_z3_success = not str(z3_result).startswith('Error')
            if is_z3_success:
                z3_ans = str(z3_result).strip().lower()
                # Option 1 Mapping: Map 'uncertain' -> 'no' for Yes/No questions
                if gt_ans in ["yes", "no", "uncertain"]:
                    if z3_ans == "uncertain" and gt_ans == "no":
                        z3_ans = "no"

                if z3_ans == gt_ans:
                    total_score += 4.0
                else:
                    total_score -= 3.0 # Severe penalty for incorrect proofs
            else:
                total_score -= 2.0 # Code crash

        avg_score = total_score / len(llm_questions) if llm_questions else -3.0

        rewards.append(avg_score)

    return rewards

def anti_fact_hallucination_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    3. Anti-Hallucination: Penalizes adding arbitrary variables as True/False (e.g. s.add(Var == True))
       without being part of logic operations. Also penalizes adding option variables directly (cheating).
    """
    rewards = []
    for completion in completions:
        parsed = parse_xml_response(completion)
        questions = parsed.get("questions", [])
        if not questions:
            rewards.append(0.0)
            continue 

        penalty = 0.0
        for q in questions:
            z3_code = q.get("z3_code", "")
            matches = re.findall(r"s\.add\(\w+\s*==\s*(True|False)\)", z3_code)
            if len(matches) > 0:
                penalty -= 1.5 * len(matches)            

            cheating_options = re.findall(r"s\.add\((Option[A-D]|best_option)", z3_code)
            if len(cheating_options) > 0:
                penalty -= 2.0 * len(cheating_options)                

        rewards.append(penalty / len(questions))

    return rewards



def anti_empty_option_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    4. Anti-Empty MCQ Options: Penalizes defining option variables as simple free Bools
       without gán constraints to them.
    """
    rewards = []

    for completion in completions:
        parsed = parse_xml_response(completion)
        questions = parsed.get("questions", [])
        if not questions:
            rewards.append(0.0)
            continue        

        penalty = 0.0
        for q in questions:
            z3_code = q.get("z3_code", "")
            if "check_mcq" in z3_code:
                for opt in ['OptionA', 'OptionB', 'OptionC', 'OptionD']:
                    if f"{opt} = Bool(" in z3_code:
                        has_logic = re.search(rf"{opt}\s*=\s*(And|Or|Implies|Not)", z3_code)
                        if not has_logic:
                            penalty -= 1.5
        rewards.append(penalty / len(questions))
    return rewards

def fol_representation_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    5. Logic representation (Propositional vs FOL): Penalizes using ONLY simple Bools
       for quantifier-heavy premises.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        parsed = parse_xml_response(completion)
        questions = parsed.get("questions", [])
        if not questions:
            rewards.append(0.0)
            continue
            
        has_quantifier = any(q in prompt.lower() for q in ['every', 'all', 'there exists', 'at least one'])
        score = 0.0

        for q in questions:
            z3_code = q.get("z3_code", "")
            if has_quantifier:
                uses_fol = "DeclareSort" in z3_code or "Function" in z3_code
                if not uses_fol and "Bool" in z3_code:
                    score -= 1.0
                elif uses_fol:
                    score += 1.0
        rewards.append(score / len(questions))

    return rewards



def premises_used_reward_func(prompts, completions, messages, **kwargs) -> list[float]:
    """
    6. Premises Used Reward: Rewards model based on F1-score of predicted premises indices
       compared to the ground truth premises_used.
    """
    rewards = []
    for completion, msg_list in zip(completions, messages):
        assistant_text = next((m['content'] for m in msg_list if m['role'] == 'assistant'), '')
        gt_parsed = parse_xml_response(assistant_text)
        gt_questions = gt_parsed.get('questions', [])
        gt_used = set(gt_questions[0].get('premises_used', [])) if gt_questions else set()

        parsed = parse_xml_response(completion)
        llm_questions = parsed.get('questions', [])
        llm_used = set(llm_questions[0].get('premises_used', [])) if llm_questions else set()

        if not gt_used:
            rewards.append(2.0 if not llm_used else 0.0)
            continue

        if not llm_used:
            rewards.append(0.0)
            continue

        intersection = len(gt_used & llm_used)
        precision = intersection / len(llm_used)
        recall = intersection / len(gt_used)
        
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0

        rewards.append(f1 * 2.0)

    return rewards

print('All 6 custom GRPO Reward & Penalty functions (including parallelized Z3) implemented successfully!')


All 6 custom GRPO Reward & Penalty functions (including parallelized Z3) implemented successfully!


In [6]:
# ========================================================
# STEP 5: Load SFT Warm-Started Model & Tokenizer
# ========================================================
print(f'Loading tokenizer for {args.model_id}...')
tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left' # Crucial for GRPO generation batching

print(f'Loading base model {args.model_id} in bfloat16...')
base_model = AutoModelForCausalLM.from_pretrained(
    args.model_id,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='flash_attention_2'
)

print(f'Applying SFT warm-start LoRA adapter from {args.sft_adapter_dir}...')
model = PeftModel.from_pretrained(base_model, args.sft_adapter_dir, is_trainable=True)
print('Warm-started LoRA active and trainable!')

Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Applying SFT warm-start LoRA adapter from NguyenAn05/qwen2.5-type1-sft-lora...


adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/646M [00:00<?, ?B/s]

Warm-started LoRA active and trainable!


In [7]:
# ========================================================
# STEP 6: Load GRPO Training Datasets
# ========================================================
print(f'Loading datasets from Hugging Face Hub: {args.dataset_id}...')
dataset = load_dataset(args.dataset_id)

def format_grpo_sample(example):
    messages = example['messages']
    system_text = next((m['content'] for m in messages if m['role'] == 'system'), '')
    user_text = next((m['content'] for m in messages if m['role'] == 'user'), '')

    prompt_text = tokenizer.apply_chat_template([
        {'role': 'system', 'content': system_text},
        {'role': 'user', 'content': user_text}
    ], tokenize=False, add_generation_prompt=True)

    return {
        'prompt': prompt_text,
        'messages': messages
    }

train_dataset = dataset['train'].map(format_grpo_sample)
val_split_name = 'validation' if 'validation' in dataset else 'test'
val_dataset = dataset[val_split_name].map(format_grpo_sample)

print(f'Training size: {len(train_dataset)}, Validation size: {len(val_dataset)}')
print('Sample GRPO prompt format:')
print(train_dataset[0]['prompt'][:500])

Loading datasets from Hugging Face Hub: NguyenAn05/exact_2026_model1_fol_translator...


README.md:   0%|          | 0.00/572 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.38M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/791k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/828k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/168 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/170 [00:00<?, ? examples/s]

Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

Map:   0%|          | 0/168 [00:00<?, ? examples/s]

Training size: 1348, Validation size: 168
Sample GRPO prompt format:
<|im_start|>system
You are a logical reasoning expert. Given a set of premises in natural language, a query, and a list of options:
1. You must first provide a concise step-by-step reasoning explanation inside the <think>...</think> tags.
2. After closing the </think> tag, output the finalized structured results using XML tags:
   - Use <premises_fol>...</premises_fol> to list the FOL translations of all premises.
   - Use <question_output>...</question_output> containing the structured output f


In [ ]:
# ========================================================
# STEP 7: Configure GRPOTrainer & Run Reinforcement Learning
# ========================================================
import inspect
from trl import GRPOConfig, GRPOTrainer

# Auto-detect if vllm is installed for AMD GPU acceleration
try:
    import vllm
    HAS_VLLM = True
    print("SUCCESS: vLLM library found! Enabling 10x generation speedup.")
except ImportError:
    HAS_VLLM = False
    print("WARNING: vLLM is not installed. Rollout generation will be slow. Run 'pip install vllm' to enable speedup.")

config_fields = GRPOConfig.__dataclass_fields__.keys()
trainer_fields = inspect.signature(GRPOTrainer.__init__).parameters.keys()

config_kwargs = {
    'output_dir': args.output_dir,
    'learning_rate': 1e-6,         # Ultra-low learning rate for stable alignment
    'beta': 0.02,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 8, # Increased from 4 to 16 for MI300X high utilization
    'gradient_accumulation_steps': 2,  # Decreased from 4 to 1 for faster steps
    'num_generations': 4,          # Group size G = 8
    'temperature': 0.9,            # High temperature for rollout diversity
    'use_vllm': HAS_VLLM,          # Enabled if vllm is installed
    'vllm_gpu_memory_utilization': 0.4, # Give VRAM back to training model (required to prevent OOM)
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.03,
    'logging_steps': 1,
    'bf16': True,
    'save_strategy': 'steps',
    'save_steps': 10,
    'save_total_limit': 1,         # Keep only 1 local checkpoint
    'push_to_hub': True,
    'hub_model_id': args.hub_model_id,
    'hub_strategy': 'checkpoint',
    'hub_token': HF_TOKEN,
    'hub_private_repo': True,
    'report_to': 'none',
    'seed': args.seed,
    'max_steps': 200
}

trainer_kwargs = {
    'model': model,
    'reward_funcs': [
        xml_format_reward_func, 
        z3_accuracy_reward_func, 
        anti_fact_hallucination_reward_func, 
        anti_empty_option_reward_func,
        fol_representation_reward_func,
        premises_used_reward_func
    ],
    'train_dataset': train_dataset,
    'eval_dataset': val_dataset,
    'processing_class': tokenizer,
    'callbacks': [GRPOLogCallback()]
}

if 'max_prompt_length' in config_fields:
    config_kwargs['max_prompt_length'] = 512
elif 'max_prompt_length' in trainer_fields:
    trainer_kwargs['max_prompt_length'] = 512

if 'max_completion_length' in config_fields:
    config_kwargs['max_completion_length'] = 1024 # Space for deep CoT thinking
elif 'max_completion_length' in trainer_fields:
    trainer_kwargs['max_completion_length'] = 1024

print('Initializing GRPOTrainer...')
training_args = GRPOConfig(**config_kwargs)
trainer = GRPOTrainer(args=training_args, **trainer_kwargs)

# Smart checkpoint resuming
last_ckpt = None

if os.path.exists(args.output_dir):
    checkpoints = [os.path.join(args.output_dir, d) for d in os.listdir(args.output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        last_ckpt = max(checkpoints, key=os.path.getmtime)
        print(f'Resuming from checkpoint: {last_ckpt}')

if not last_ckpt:
    try:
        from huggingface_hub import HfApi, snapshot_download
        from transformers.trainer_utils import get_last_checkpoint
        api = HfApi()
        api.model_info(args.hub_model_id, token=HF_TOKEN)
        print(f'Hugging Face Hub repo found: {args.hub_model_id}. Pulling checkpoint files...')
        snapshot_download(
            repo_id=args.hub_model_id,
            token=HF_TOKEN,
            local_dir=args.output_dir,
        )
        last_ckpt = get_last_checkpoint(args.output_dir)
        if last_ckpt is None:
            hub_ckpt = os.path.join(args.output_dir, 'last-checkpoint')
            if os.path.isdir(hub_ckpt):
                last_ckpt = hub_ckpt
        print(f'Successfully resumed from HUB checkpoint: {last_ckpt}')
    except Exception as e:
        print(f'Hub pull skipped ({e}). Starting fresh.')


print('\n--- Starting GRPO Reinforcement Learning Training ---')

trainer.train(resume_from_checkpoint=last_ckpt)

print('Training complete!')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Initializing GRPOTrainer...
Hugging Face Hub repo found: NguyenAn05/qwen2.5-type1-grpo-lora. Pulling checkpoint files...


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Successfully resumed from HUB checkpoint: models/qwen2.5-type1-grpo-lora/last-checkpoint

--- Starting GRPO Reinforcement Learning Training ---


In [ ]:
# ========================================================
# STEP 8: Push Final Aligned Model & Cleanup
# ========================================================
hf_target_repo = args.hub_model_id
print(f'Final push of aligned model to Hugging Face Hub: {hf_target_repo}...')
trainer.push_to_hub()
tokenizer.push_to_hub(hf_target_repo, private=True)
print(f'Success! Aligned GRPO adapter published to {hf_target_repo}!')

# Clean up local checkpoint directory to free disk space
import shutil
if os.path.exists(args.output_dir):
    shutil.rmtree(args.output_dir, ignore_errors=True)
    print(f'Local directory {args.output_dir} cleaned up.')